In [1]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shuow\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shuow\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
from segmenters.rules import split_numbered_list
test = "For a GS-13 position you must have served 52 weeks at the GS-12. The grade may have been in any occupation."
print(split_numbered_list(test))

['For a GS-13 position you must have served 52 weeks at the GS-12. The grade may have been in any occupation.']


In [3]:
import time
import spacy
import nltk
import pandas as pd

df = pd.read_json("data/raw/sample/sample.jsonl", lines=True)
df_sample = df["text"].dropna().sample(10, random_state=42)
nlp = spacy.load("en_core_web_sm")

from preprocessing.cleaner import clean_description
from segmenters.nltk_splitter import nltk_split

# 取10条样本测试速度
df_sample = df["text"].dropna().sample(10, random_state=42)
cleaned_samples = [clean_description(raw) for raw in df_sample]

# 测试NLTK上游速度
start = time.time()
for text in cleaned_samples:
    nltk_split(text)
nltk_time = time.time() - start
print(f"NLTK 10条: {nltk_time:.2f}秒 → 预计2万条: {nltk_time/10*20000/60:.1f}分钟")

# 测试spaCy句子切割速度（只用sents，不做依存分析）
start = time.time()
for text in cleaned_samples:
    doc = nlp(text)
    sents = [s.text for s in doc.sents]
spacy_sent_time = time.time() - start
print(f"spaCy分句 10条: {spacy_sent_time:.2f}秒 → 预计2万条: {spacy_sent_time/10*20000/60:.1f}分钟")

# 测试spaCy完整分析速度（包含依存解析）
start = time.time()
for text in cleaned_samples:
    doc = nlp(text)
    for sent in doc.sents:
        for token in sent:
            _ = token.dep_
spacy_full_time = time.time() - start
print(f"spaCy完整分析 10条: {spacy_full_time:.2f}秒 → 预计2万条: {spacy_full_time/10*20000/60:.1f}分钟")

NLTK 10条: 0.06秒 → 预计2万条: 2.1分钟
spaCy分句 10条: 1.60秒 → 预计2万条: 53.2分钟
spaCy完整分析 10条: 1.36秒 → 预计2万条: 45.4分钟


In [4]:
import pandas as pd
import nltk
import re
from preprocessing.cleaner import clean_description
from segmenters.spacy_segmenter import SpacySegmenter
from segmenters.nltk_splitter import nltk_split

CLEAN_PATTERN = re.compile(r"(;|\s\+\s|\s\*\s|\s\-\s|\s•\s|\s·\s|--|\*\*)")

def original_nltk_split(text: str) -> list[str]:
    text = ". ".join(text.split("\n"))
    text = CLEAN_PATTERN.sub(". ", text)
    text = text.replace("\n", ".").replace("..", ".")
    return nltk.sent_tokenize(text)

segmenter = SpacySegmenter()

df = pd.read_json("data/raw/sample/sample.jsonl", lines=True)
raw = df["text"].dropna().sample(1, random_state=42).iloc[0]
cleaned = clean_description(raw)

# 两种方法都用cleaned text
original_result = original_nltk_split(cleaned)
our_result = segmenter.segment(cleaned)

print(f"原始NLTK: {len(original_result)}句")
print(f"我们的方法: {len(our_result)}句")

# 找出被我们切分的句子（原来是一句，现在变成多句）
our_set = set(s.strip(" .") for s in our_result)

print(f"\n=== 被我们进一步切分的句子 ===")
for orig in original_result:
    orig_clean = orig.strip(" .")
    # 检查原句是否被切成了多个片段
    matching_parts = [s for s in our_result if s.strip(" .") in orig_clean or orig_clean in s.strip(" .")]
    if len(matching_parts) > 1:
        print(f"\n原句: {orig}")
        for p in matching_parts:
            print(f"  → {p}")

原始NLTK: 89句
我们的方法: 92句

=== 被我们进一步切分的句子 ===

原句: accurately and efficiently vital signs, obtaining specimens, and supporting the Physician with any other diagnostic tests.
  → accurately and efficiently vital signs, obtaining specimens
  → supporting the Physician with any other diagnostic tests.

原句: Applies prosthetic devices, assists in resident therapies such as Occupational Therapy (OT), Kinesiotherapy (OT), Speech, Physical Therapy (PT) and Recreation.
  → Applies prosthetic devices
  → assists in resident therapies such as Occupational Therapy (OT), Kinesiotherapy (OT), Speech, Physical Therapy (PT) and Recreation.

原句: For example, recognizing need for basic life support, controlling bleeding and assisting with behavior crisis, etc.
  → For example, recognizing need for basic life support
  → controlling bleeding and assisting with behavior crisis, etc.


In [5]:
from segmenters.nltk_splitter import nltk_split

cleaned_sentences = nltk_split(cleaned)

missing = [
    "GS-5 Level",
    "OR Education",
    "This credited service",
    "Such credit",
]

for keyword in missing:
    matches = [s for s in cleaned_sentences if keyword in s]
    if matches:
        print(f"✓ '{keyword}' 在nltk_split里:")
        for m in matches:
            print(f"  {m[:120]}")
    else:
        print(f"✗ '{keyword}' 在nltk_split里没有找到")

✓ 'GS-5 Level' 在nltk_split里:
  GS-5 Level: Experience: One year of progressively responsible assignments and experience equivalent to the GS-4 level wh
✓ 'OR Education' 在nltk_split里:
  OR Education: Successful completion of a 4-year course of study above high school leading to a bachelor's degree that in
✓ 'This credited service' 在nltk_split里:
  This credited service can be used in determining the rate at which they earn annual leave.
✓ 'Such credit' 在nltk_split里:
  Such credit must be requested and approved prior to the appointment date and is not guaranteed.


In [6]:
raw = df["text"].dropna().sample(1, random_state=42).iloc[0]

# 方法1：原始NLTK直接对raw text
original_result = original_nltk_split(raw)

# 方法2：我们的方法，先clean再segment
cleaned = clean_description(raw)
our_result = segmenter.segment(cleaned)

print(f"原始NLTK（对raw text）: {len(original_result)}句")
print(f"我们的方法（clean+segment）: {len(our_result)}句")

# 找出原始NLTK有但我们没有的
our_normalized = set(s.strip(" .").lower() for s in our_result)

print(f"\n=== 原始NLTK有但我们过滤掉的===")
count = 0
for s in original_result:
    normalized = s.strip(" .").lower()
    if normalized not in our_normalized:
        print(f"  ✗ {s[:120]}")
        count += 1

原始NLTK（对raw text）: 240句
我们的方法（clean+segment）: 92句

=== 原始NLTK有但我们过滤掉的===
  ✗ USAJOBS.
  ✗ Job Announcement family-of-overseas-employees-icon federal-employees-competitive-service-icon federal-employees-competit
  ✗ internal-to-an-agency-icon Created with Sketch.
  ✗ internal-to-an-agency-icon Created with Sketch.
  ✗ military-spouses-icon Created with Sketch.
  ✗ national-guard-and-reserves-icon Created with Sketch.
  ✗ native-americans-icon Created with Sketch.
  ✗ peace-corps-and-americorps-icons Open-to-the-public-icon senior-executive-service-icon se-other students-icon recent-gra
  ✗ Help Overview Hiring complete Open & closing dates 12/29/2023 to 01/09/2024 Salary $33,905.
  ✗ $54,752 per year Pay scale & grade GS 3.
  ✗ 5 Location 1 vacancy in the following location: Danville, IL 1 vacancy Remote job No Telework eligible No Travel Require
  ✗ accurately and efficiently vital signs, obtaining specimens, and supporting the Physician with any other diagnostic test
  ✗ Applies prost

In [8]:
import importlib
import segmenters.spacy_utils as utils_module
import segmenters.spacy_segmenter as seg_module
importlib.reload(utils_module)
importlib.reload(seg_module)

from segmenters.spacy_segmenter import SpacySegmenter
from preprocessing.cleaner import clean_description
from segmenters.nltk_splitter import nltk_split
import nltk
import re

CLEAN_PATTERN = re.compile(r"(;|\s\+\s|\s\*\s|\s\-\s|\s•\s|\s·\s|--|\*\*)")

def original_nltk_split(text):
    text = ". ".join(text.split("\n"))
    text = CLEAN_PATTERN.sub(". ", text)
    text = text.replace("\n", ".").replace("..", ".")
    return nltk.sent_tokenize(text)

segmenter = SpacySegmenter()

df_sample = df["text"].dropna().sample(5, random_state=123)

for i, raw in enumerate(df_sample):
    cleaned = clean_description(raw)
    nltk_only = nltk_split(cleaned)
    final = segmenter.segment(cleaned)
    original = original_nltk_split(raw)
    
    print(f"\n{'='*70}")
    print(f"样本 {i+1} | 原始NLTK:{len(original)}句 | clean+NLTK:{len(nltk_only)}句 | clean+NLTK+spaCy:{len(final)}句")
    print(f"{'='*70}")
    
    nltk_set = set(s.strip(" .") for s in nltk_only)
    spacy_splits = []
    for orig_sent in nltk_only:
        parts = [s for s in final if s.strip(" .") not in nltk_set and s.strip(" .") in orig_sent]
        if parts:
            spacy_splits.append((orig_sent, parts))
    
    if spacy_splits:
        print(f"\n--- spaCy进一步切分的句子 ---")
        for orig, parts in spacy_splits:
            print(f"\n  原句: {orig}")
            for p in parts:
                print(f"    → {p}")
    else:
        print(f"\n  （无进一步切分）")


样本 1 | 原始NLTK:238句 | clean+NLTK:53句 | clean+NLTK+spaCy:54句

--- spaCy进一步切分的句子 ---

  原句: If you have retired from federal service and are interested in employment as a reemployed annuitant, see the information in the Reemployed Annuitant information sheet via To Remain An Active Duty USPHS Commissioned Corps Officer Join more than 6,500 highly qualified public health professionals as part of the U.S. Public Health Service.
    → If you have retired from federal service and are interested in employment as a reemployed annuitant
    → see the information in the Reemployed Annuitant information sheet via To Remain An Active Duty USPHS Commissioned Corps Officer Join more than 6,500 highly qualified public health professionals as part of the U.S. Public Health Service.

样本 2 | 原始NLTK:250句 | clean+NLTK:100句 | clean+NLTK+spaCy:103句

--- spaCy进一步切分的句子 ---

  原句: The Licensed Vocational Nurse is responsible for observing, recording, and reporting changes in the patient's condition, as well as